In [2]:
# import statements 
# --taken from class logistic regression collab
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display
from sklearn.base import clone
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, GroupKFold, GridSearchCV
from sklearn.metrics import log_loss, confusion_matrix, accuracy_score, roc_curve, roc_auc_score

px.defaults.template = "plotly_white"
px.defaults.color_discrete_sequence = px.colors.qualitative.Dark24

In [3]:
#using the train_users_cleaned
train_users=pd.read_csv('train_users_cleaned.csv')
#train_users.head()


In [4]:
# NOTE need to do some more data cleaning
#converting url to 1,0 value if has url or not
## do the same for if a location is listed or if has a description
train_users['url']=train_users['url'].notna().astype(int)
train_users['description']=train_users['description'].notna().astype(int)
train_users['location']=train_users['location'].notna().astype(int)


Preprocessing Data

In [5]:
train_users.columns

Index(['ID', 'account_type', 'name', 'screen_name', 'location', 'description',
       'url', 'followers_count', 'friends_count', 'listed_count', 'created_at',
       'favourites_count', 'geo_enabled', 'verified', 'statuses_count',
       'is_translation_enabled', 'profile_background_tile',
       'profile_use_background_image', 'has_extended_profile',
       'default_profile', 'default_profile_image', 'domain', 'domain_politics',
       'domain_business', 'domain_entertainment', 'domain_sports', 'n_tweets'],
      dtype='str')

In [6]:
# setting y - what we want to predict
y= train_users['account_type']

#setting x - predictors
x=train_users[['location', 'description', 'url', 'followers_count', 'friends_count',
              'listed_count','favourites_count', 'geo_enabled', 'verified',
              'statuses_count','is_translation_enabled', 'profile_background_tile',
              'profile_use_background_image', 'has_extended_profile','default_profile',
              'default_profile_image', 'domain_politics','domain_business',
              'domain_entertainment', 'domain_sports', 'n_tweets']]

In [7]:
# for test set:

test_users=pd.read_csv('test_users_cleaned.csv')

test_users['url']=test_users['url'].notna().astype(int)
test_users['descrition']=test_users['description'].notna().astype(int)


y_test= test_users['account_type']

x_test=test_users[['location', 'description', 'url', 'followers_count', 'friends_count',
              'listed_count','favourites_count', 'geo_enabled', 'verified',
              'statuses_count','is_translation_enabled', 'profile_background_tile',
              'profile_use_background_image', 'has_extended_profile','default_profile',
              'default_profile_image', 'domain_politics','domain_business',
              'domain_entertainment', 'domain_sports', 'n_tweets']]


In [8]:
y=(train_users["account_type"] == "bot").astype(int)        # bot = 1, human = 0
y_test=(test_users["account_type"] == "bot").astype(int)

In [9]:
## split train set into training and validation set
x, x_valid, y, y_valid  = train_test_split(x, y, test_size=0.25, random_state=6021)



In [10]:
# categorical and numerical variables 
## NOTE : not using date columns !!!

categorical= ['location', 'description', 'url','geo_enabled', 'verified',
              'is_translation_enabled','profile_use_background_image',
              'has_extended_profile','default_profile','default_profile_image',
              'domain_politics','domain_business','domain_entertainment','domain_sports']
numerical= ['followers_count', 'friends_count', 'listed_count','favourites_count',
            'statuses_count','profile_background_tile','n_tweets']

In [11]:
#preprocessing
preprocess = ColumnTransformer(transformers=[
    ("encoder", OneHotEncoder(drop="first", handle_unknown="ignore"), categorical),
    ("numeric", StandardScaler(), numerical)
])

In [12]:
#logistic regression on test data
logreg = LogisticRegression(max_iter=5000, random_state=123)
pipe = Pipeline(steps=[("preprocess", preprocess), ("model", logreg)])
pipe.fit(x, y)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocess', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](21,)","['location','description','url',...,'domain_entertainment','domain_sports', 'n_tweets']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,21
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('encoder', ...), ('numeric', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder

In [13]:
# probabilitys on training versus test data
p_train = pipe.predict_proba(x)[:, 1]
p_test = pipe.predict_proba(x_test)[:, 1]

#log odds with 0.5 decision threshold
log_odds_train = pipe.decision_function(x)
log_odds_test = pipe.decision_function(x_test)
threshold = 0.5

/opt/miniconda3/envs/ds6001/lib/python3.14/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [0, 1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/opt/miniconda3/envs/ds6001/lib/python3.14/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [0, 1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


In [14]:
#what are the coefficiants on each predictor
print("Intercept:", logreg.intercept_[0])

coef = pd.DataFrame({
    "Feature": pipe.named_steps["preprocess"].get_feature_names_out(),
##used AI for feature code modification
    "Coefficient": logreg.coef_[0],
    "Odds Ratio": np.exp(logreg.coef_[0])
})

coef

Intercept: 0.9802612365596944


,Feature,Coefficient,Odds Ratio
0,encoder__location_1,0.144383,1.155327
1,encoder__description_1,-0.046482,0.954582
2,encoder__url_1,0.400156,1.492058
3,encoder__geo_enabled_True,-0.383167,0.681699
4,encoder__verified_True,-6.547110,0.001434
5,encoder__is_translation_enabled_True,-0.075106,0.927645
6,encoder__profile_use_background_image_True,-0.224027,0.799293
7,encoder__has_extended_profile_True,-0.526138,0.590882
8,encoder__default_profile_True,0.101687,1.107037
9,encoder__default_profile_image_True,0.514700,1.673137


In [15]:
#confusion matrix
y_pred = (p_test >= threshold).astype(int)

cm = confusion_matrix(y_test, y_pred, labels=[1, 0])

cm_df = pd.DataFrame(
    cm,
    index=["Human", "Bot"],
    columns=["Human", "Bot"]
)

cm_df

,Human,Bot
Human,639,1
Bot,212,331


In [16]:
#confusion matrix visualiztation
fig = px.imshow(
    cm_df,
    text_auto=True,
    color_continuous_scale="Blues",
    labels=dict(x="Predicted", y="Actual", color="Count"),
    title="Human/Bot Confusion Matrix"
)

fig.update_layout(
    xaxis_title="Predicted Outcome",
    yaxis_title="Actual Outcome"
)

fig.show()

In [17]:
#looking at results
comparison = pd.DataFrame({
    "Set": ["Train", "Test"],
    "Log loss": [
        log_loss(y, p_train),
        log_loss(y_test, p_test),
    ],
    "Accuracy": [
        accuracy_score(y, p_train >= threshold),
        accuracy_score(y_test, p_test >= threshold)
    ]
})

print("\nTraining vs test metrics:")
print(comparison.round(3).to_string(index=False))


Training vs test metrics:
  Set  Log loss  Accuracy
Train     0.407     0.811
 Test     0.399     0.820


In [18]:
#AUC-ROC
thresholds = [0.1, 0.3, 0.5, 0.7, 0.9]

auc = roc_auc_score(y_test, p_test)
ll = log_loss(y_test, p_test)

test_rows = []

for t in thresholds:
    y_pred = (p_test >= t).astype(int)

    tp, fn, fp, tn = confusion_matrix(
        y_test, y_pred, labels=[1, 0]
    ).ravel()

    tpr = tp / (tp + fn)
    fpr = fp / (fp + tn)
    precision = tp / (tp + fp) if tp + fp > 0 else 0
    accuracy = (tp + tn) / (tp + fp + fn + tn)
    f1 = 2 * tp / (2 * tp + fp + fn) if 2 * tp + fp + fn > 0 else 0

    test_rows.append({
        "Threshold": t,
        "TP": tp, "FP": fp, "FN": fn, "TN": tn,
        "TPR": tpr, "FPR": fpr,
        "Precision": precision, "Accuracy": accuracy, "F1": f1,
        "Total cost": fp + fn,
        "Log loss": ll, "ROC AUC": auc
    })

test_results = pd.DataFrame(test_rows)
display(test_results.round(3))

,Threshold,TP,FP,FN,TN,TPR,FPR,Precision,Accuracy,F1,Total cost,Log loss,ROC AUC
0,0.1,640,217,0,326,1.000,0.400,0.747,0.817,0.855,217,0.399,0.848
1,0.3,640,217,0,326,1.000,0.400,0.747,0.817,0.855,217,0.399,0.848
2,0.5,639,212,1,331,0.998,0.390,0.751,0.820,0.857,213,0.399,0.848
3,0.7,441,117,199,426,0.689,0.215,0.790,0.733,0.736,316,0.399,0.848
4,0.9,6,1,634,542,0.009,0.002,0.857,0.463,0.019,635,0.399,0.848


In [19]:
#AUC-ROC Curve
fpr_curve, tpr_curve, roc_thresholds = roc_curve(y_test, p_test)

roc_data = pd.DataFrame({
    "FPR": fpr_curve,
    "TPR": tpr_curve
})

fig = px.line(
    roc_data, x="FPR", y="TPR",
    title=f"Human/Bot ROC Curve | AUC = {auc:.3f}"
)

fig.add_scatter(
    x=test_results["FPR"],
    y=test_results["TPR"],
    mode="markers+text",
    text=[f"t = {t:.1f}" for t in thresholds],
    textposition="bottom right",
    marker=dict(size=10),
    name="Five thresholds"
)

fig.add_scatter(
    x=[0, 1], y=[0, 1],
    mode="lines", line=dict(dash="dash", color="gray"),
    name="Random classification"
)

fig.update_xaxes(range=[0, 1], title="False Positive Rate (FPR)")
fig.update_yaxes(range=[0, 1], title="True Positive Rate (TPR)")
fig.show()

In [20]:
# Ridge and Lasso regularization + Coefficiant path Graph
C_values = [0.02, 0.05, 0.1, 0.3, 1, 10]
coef_rows = []

for method in ["Ridge (L2)", "Lasso (L1)"]:
    for C in C_values:
        regularized = Pipeline([
            ("preprocess", clone(preprocess)),
            ("model", LogisticRegression(
                l1_ratio=0 if method == "Ridge (L2)" else 1,
                C=C, solver="liblinear", max_iter=5000, tol=1e-8, random_state=123
            ))
        ])
        regularized.fit(x, y)

        names = regularized.named_steps["preprocess"].get_feature_names_out()
        betas = regularized.named_steps["model"].coef_[0]

        for name, beta in zip(names, betas):
            coef_rows.append({"Method": method, "C": C,
                              "Feature": name.split("__")[-1],
                              "Coefficient": beta})

coef_paths = pd.DataFrame(coef_rows)

for method in ["Ridge (L2)", "Lasso (L1)"]:
    fig = px.line(
        coef_paths[coef_paths["Method"] == method],
        x="C", y="Coefficient", color="Feature", markers=True,
        log_x=True, title=f"Twitter Bots: {method} Coefficient Paths"
    )
    fig.add_hline(y=0, line_dash="dash")
    fig.show()

In [21]:
# number of non-zero coefficaints present
coef_paths["Nonzero"] = coef_paths["Coefficient"].abs() > 1e-6
demo_counts = coef_paths.groupby(["C", "Method"])["Nonzero"].sum().unstack()
display(demo_counts.rename_axis(columns=None))

,Lasso (L1),Ridge (L2)
C,,
0.02,10,21
0.05,10,21
0.10,15,21
0.30,18,21
1.00,19,21
10.00,21,21


In [22]:
# sets base -- no coefficiant penalty
full_model = Pipeline([
    ("preprocess", clone(preprocess)),
    ("model", LogisticRegression(
        C=np.inf, solver="lbfgs", max_iter=5000, tol=1e-8, random_state=123
    ))
])

In [23]:
ridge = Pipeline([
    ("preprocess", clone(preprocess)),
    ("model", LogisticRegression(
        l1_ratio=0, solver="liblinear", max_iter=5000, tol=1e-8, random_state=123
    ))
])

lasso = Pipeline([
    ("preprocess", clone(preprocess)),
    ("model", LogisticRegression(
        l1_ratio=1, solver="liblinear", max_iter=5000, tol=1e-8, random_state=123
    ))
])

#applys coefficiant penalties to regularizations

C_grid = np.unique(np.r_[np.logspace(-4, 3, 36), C_values])
cv = GroupKFold(n_splits=5, shuffle=True, random_state=6021)

ridge_cv = GridSearchCV(
    ridge, {"model__C": C_grid}, cv=cv, scoring="neg_log_loss", error_score="raise"
)
ridge_cv.fit(x, y)

lasso_cv = GridSearchCV(
    lasso, {"model__C": C_grid}, cv=cv, scoring="neg_log_loss", error_score="raise"
)
lasso_cv.fit(x, y)

In [24]:
full_model.fit(x,y)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocess', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](21,)","['location','description','url',...,'domain_entertainment','domain_sports', 'n_tweets']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,21
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('encoder', ...), ('numeric', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder

In [25]:
##AI bc not have item to group on in code from class
from sklearn.model_selection import StratifiedKFold

C_grid = np.unique(np.r_[np.logspace(-4, 3, 36), C_values])
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=6021)

ridge_cv = GridSearchCV(
    ridge, {"model__C": C_grid}, cv=cv, scoring="neg_log_loss", error_score="raise"
)
ridge_cv.fit(x, y)

lasso_cv = GridSearchCV(
    lasso, {"model__C": C_grid}, cv=cv, scoring="neg_log_loss", error_score="raise"
)
lasso_cv.fit(x, y)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step... tol=1e-08))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.",{'model__C': array([1.0000...00000000e+03])}
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'neg_log_loss'
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",StratifiedKFo... shuffle=True)
,"error_score error_score: 'raise' or numeric, default=np.nanValue to assign to the score if an error occurs in estimator fitting.If set to 'raise', the error is raised. If a numeric value is given,FitFailedWarning is raised. This parameter does not affect the refitstep, which will always raise the error.",'raise'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevi

In [26]:
#selected coeff
ridge_model = ridge_cv.best_estimator_
lasso_model = lasso_cv.best_estimator_

best_C_ridge = ridge_cv.best_params_["model__C"]
best_C_lasso = lasso_cv.best_params_["model__C"]

models = {"All 21 (unpenalized)": full_model,
          "Ridge": ridge_model, "Lasso": lasso_model}
searches = {"Ridge": ridge_cv, "Lasso": lasso_cv}

display(pd.DataFrame({
    "Model": ["Ridge", "Lasso"],
    "Selected C": [best_C_ridge, best_C_lasso],
    "Mean CV log loss": [-ridge_cv.best_score_, -lasso_cv.best_score_]
}).round(6))

,Model,Selected C,Mean CV log loss
0,Ridge,3.981072,0.411980
1,Lasso,0.398107,0.411063


In [27]:
cv_tables = []
for name, search in searches.items():
    results = pd.DataFrame(search.cv_results_)
    cv_tables.append(pd.DataFrame({
        "Model": name,
        "C": results["param_model__C"].astype(float),
        "CV log loss": -results["mean_test_score"],
        "Fold SD": results["std_test_score"]
    }))

cv_results = pd.concat(cv_tables, ignore_index=True)

In [28]:
for name, search in searches.items():
    chosen_C = search.best_params_["model__C"]
    fig = px.line(
        cv_results[cv_results["Model"] == name],
        x="C", y="CV log loss", log_x=True, markers=True,
        title=f"{name}: five-fold CV log loss | selected C = {chosen_C:g}"
    )
    fig.add_vline(x=chosen_C, line_dash="dash", line_color="black")
    fig.update_xaxes(tickvals=[0.0001, 0.001, 0.01, 0.1, 1, 10, 100, 1000],
                     ticktext=["0.0001", "0.001", "0.01", "0.1", "1", "10", "100", "1000"])
    fig.show()

In [29]:
import sklearn
from packaging.version import Version
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

C_lasso = 0.01

if Version(sklearn.__version__) >= Version("1.8"):
    lasso_options = {"l1_ratio": 1}
else:
    lasso_options = {"penalty": "l1"}

lasso_model = Pipeline([
    ("preprocess", clone(preprocess)),
    ("model", LogisticRegression(
        C=C_lasso,
        solver="liblinear",
        max_iter=5000,
        tol=1e-8,
        random_state=123,
        **lasso_options
    ))
])

lasso_model.fit(x, y)

models["Lasso"] = lasso_model

print("scikit-learn version:", sklearn.__version__)
print("Lasso C:", lasso_model.named_steps["model"].C)

scikit-learn version: 1.9.0
Lasso C: 0.01


In [30]:
#which features does the lasso drop

feature_names = full_model.named_steps["preprocess"].get_feature_names_out()
feature_names = [name.split("__")[-1] for name in feature_names]

coefficients = pd.DataFrame({"Feature": feature_names})
model_rows = []
for name, model in models.items():
    beta = model.named_steps["model"].coef_[0]
    coefficients[name] = beta
    model_rows.append({"Model": name, "Nonzero slopes": int((abs(beta) > 1e-6).sum())})

feature_counts = pd.DataFrame(model_rows)
display(coefficients.round(6))
display(feature_counts)
print("Lasso drops:", coefficients.loc[abs(coefficients["Lasso"]) <= 1e-6, "Feature"].tolist())

,Feature,All 21 (unpenalized),Ridge,Lasso
0,location_1,0.141746,0.143425,0.000000
1,description_1,-0.046077,-0.044987,0.000000
2,url_1,0.411791,0.410385,0.000000
3,geo_enabled_True,-0.386224,-0.384836,0.000000
4,verified_True,-8.967636,-7.663445,-3.534961
5,is_translation_enabled_True,-0.083555,-0.085736,0.000000
6,profile_use_background_image_True,-0.222681,-0.218712,0.000000
7,has_extended_profile_True,-0.530747,-0.529092,0.000000
8,default_profile_True,0.101235,0.099609,0.000000
9,default_profile_image_True,0.532452,0.528191,0.000000


,Model,Nonzero slopes
0,All 21 (unpenalized),21
1,Ridge,21
2,Lasso,3


Lasso drops: ['location_1', 'description_1', 'url_1', 'geo_enabled_True', 'is_translation_enabled_True', 'profile_use_background_image_True', 'has_extended_profile_True', 'default_profile_True', 'default_profile_image_True', 'domain_politics_1', 'domain_business_1', 'domain_entertainment_1', 'followers_count', 'friends_count', 'listed_count', 'statuses_count', 'profile_background_tile', 'n_tweets']


In [31]:
C_path = np.unique(np.r_[C_grid, C_values])
path_rows = []

for name, search in searches.items():
    for C in C_path:
        path_model = clone(search.best_estimator_).set_params(model__C=C)
        path_model.fit(x, y)
        beta = path_model.named_steps["model"].coef_[0]
        for feature, value in zip(feature_names, beta):
            path_rows.append({"Model": name, "C": C,
                              "Feature": feature, "Coefficient": value})

paths = pd.DataFrame(path_rows)

In [32]:
for name, search in searches.items():
    chosen_C = search.best_params_["model__C"]
    fig = px.line(
        paths[paths["Model"] == name], x="C", y="Coefficient",
        color="Feature", log_x=True,
        category_orders={"Feature": feature_names},
        title=f"{name}: coefficient paths | CV-selected C = {chosen_C:g}"
    )
    fig.add_hline(y=0, line_dash="dot", line_color="gray")
    fig.add_vline(x=chosen_C, line_dash="dash", line_color="black")
    fig.update_layout(height=500)
    fig.update_xaxes(tickvals=[0.0001, 0.001, 0.01, 0.1, 1, 10, 100, 1000],
                     ticktext=["0.0001", "0.001", "0.01", "0.1", "1", "10", "100", "1000"])
    fig.show()

In [33]:
sparsity_rows = []
for C in [0.01, 0.02, best_C_lasso]:
    demo_lasso = clone(lasso_model).set_params(model__C=C).fit(x, y)
    beta = demo_lasso.named_steps["model"].coef_[0]
    grid_match = cv_results[(cv_results["Model"] == "Lasso") & np.isclose(cv_results["C"], C)]
    cv_loss = grid_match["CV log loss"].iloc[0] if len(grid_match) else np.nan
    sparsity_rows.append({"C": C, "Nonzero slopes": int((abs(beta) > 1e-6).sum()),
                          "Mean CV log loss": cv_loss,
                          "Role": "CV optimum" if C == best_C_lasso else "Stronger-penalty example"})

display(pd.DataFrame(sparsity_rows).round(6))

,C,Nonzero slopes,Mean CV log loss,Role
0,0.010000,3,0.451234,Stronger-penalty example
1,0.020000,10,0.430747,Stronger-penalty example
2,0.398107,19,0.411063,CV optimum


In [34]:
#finding optimal threshold

cost_fp, cost_fn = 3, 1
thresholds = np.round(np.arange(0.00, 1.01, 0.01), 2)
p_val = {name: model.predict_proba(x_valid)[:, 1] for name, model in models.items()}

threshold_rows = []
for name, probabilities in p_val.items():
    for t in thresholds:
        predicted = (probabilities >= t).astype(int)
        tp, fn, fp, tn = confusion_matrix(y_valid, predicted, labels=[1, 0]).ravel()
        threshold_rows.append({"Model": name, "Threshold": t,
                               "Validation cost": cost_fp * fp + cost_fn * fn,
                               "Validation accuracy": (tp + tn) / len(y_valid)})

validation_results = pd.DataFrame(threshold_rows)

In [35]:
best_thresholds = {}
selected_rows = []

for name in models:
    results = validation_results[validation_results["Model"] == name].copy()
    results["Distance from 0.5"] = abs(results["Threshold"] - 0.5)
    best = results.sort_values(["Validation cost", "Distance from 0.5", "Threshold"]).iloc[0]
    best_thresholds[name] = float(best["Threshold"])
    selected_rows.append(best)

threshold_summary = pd.DataFrame(selected_rows).drop(columns="Distance from 0.5")
display(threshold_summary.round(4))

,Model,Threshold,Validation cost,Validation accuracy
77,All 21 (unpenalized),0.77,881,0.6961
178,Ridge,0.77,878,0.6966
272,Lasso,0.70,1012,0.5923


In [36]:
fig = px.line(
    validation_results, x="Threshold", y="Validation cost", color="Model",
    title="Choose thresholds on validation data: cost = 3 × FP + FN"
)
for name, t in best_thresholds.items():
    point = threshold_summary[threshold_summary["Model"] == name]
    fig.add_scatter(x=[t], y=point["Validation cost"], mode="markers",
                    marker=dict(size=11, symbol="diamond"),
                    name=f"{name}: selected t = {t:.2f}")
fig.show()

In [37]:
p_test = {name: model.predict_proba(x_test)[:, 1] for name, model in models.items()}

test_comparison = pd.DataFrame({
    "Model": list(models),
    "Test log loss": [log_loss(y_test, p_test[name]) for name in models],
    "Nonzero slopes": feature_counts["Nonzero slopes"],
    "Selected threshold": [best_thresholds[name] for name in models]
})
display(test_comparison.round(6))

/opt/miniconda3/envs/ds6001/lib/python3.14/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [0, 1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/opt/miniconda3/envs/ds6001/lib/python3.14/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [0, 1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/opt/miniconda3/envs/ds6001/lib/python3.14/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [0, 1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


,Model,Test log loss,Nonzero slopes,Selected threshold
0,All 21 (unpenalized),0.397433,21,0.77
1,Ridge,0.398001,21,0.77
2,Lasso,0.429967,3,0.70


In [38]:
logloss_figure = px.bar(
    test_comparison, x="Model", y="Test log loss", text_auto=".4f",
    title="Test log loss — lower is better"
)
logloss_figure.update_traces(marker_color="#234C6A", width=0.55, textposition="outside")
logloss_figure.update_layout(showlegend=False, xaxis_title=None, height=420)
logloss_figure.update_xaxes(showgrid=False)
logloss_figure.update_yaxes(showgrid=False, range=[0, test_comparison["Test log loss"].max() * 1.16])
logloss_figure.show()

In [39]:
model_names = list(models)
labels = ["Human", "Bot"]
colors = [[0, "#F6F9FC"], [1, "#9EC1DF"]]
titles = ([f"{name}<br>t = 0.50" for name in model_names] +
          [f"{name}<br>selected t = {best_thresholds[name]:.2f}" for name in model_names])
selected_cm_figure = make_subplots(rows=2, cols=3, subplot_titles=titles,
                                  vertical_spacing=0.25, horizontal_spacing=0.10)

for row in [1, 2]:
    for col, name in enumerate(model_names, start=1):
        t = 0.50 if row == 1 else best_thresholds[name]
        cm = confusion_matrix(y_test, p_test[name] >= t, labels=[1, 0])
        selected_cm_figure.add_trace(go.Heatmap(
            z=cm, x=labels, y=labels, zmin=0, zmax=int(y_test.value_counts().max()),
            colorscale=colors, showscale=False, texttemplate="%{z}",
            textfont=dict(color="#132F4C", size=18),
            hovertemplate="Actual %{y}<br>Predicted %{x}<br>Count %{z}<extra></extra>"
        ), row=row, col=col)

selected_cm_figure.update_xaxes(title_text="Predicted")
selected_cm_figure.update_yaxes(title_text="Actual", autorange="reversed")
selected_cm_figure.update_layout(template="plotly_white", height=690,
                                title="Test confusion matrices: default and validation-selected thresholds")
selected_cm_figure.show()

In [40]:
metric_rows = []
for name in model_names:
    show_thresholds = sorted(set([0.10, 0.30, 0.50, 0.70, 0.90, best_thresholds[name]]))
    for t in show_thresholds:
        tp, fn, fp, tn = confusion_matrix(y_test, p_test[name] >= t, labels=[1, 0]).ravel()
        metric_rows.append({
            "Model": name, "Threshold": t, "Selected on validation": t == best_thresholds[name],
            "TP": tp, "FN": fn, "FP": fp, "TN": tn,
            "Accuracy": (tp + tn) / len(y_test), "Recall": tp / (tp + fn),
            "Precision": tp / (tp + fp) if tp + fp else 0,
            "Cost": cost_fp * fp + cost_fn * fn,
            "Log loss": log_loss(y_test, p_test[name])
        })

test_threshold_results = pd.DataFrame(metric_rows)
display(test_threshold_results.round(4))

,Model,Threshold,Selected on validation,TP,FN,FP,TN,Accuracy,Recall,Precision,Cost,Log loss
0,All 21 (unpenalized),0.10,False,640,0,217,326,0.8166,1.0000,0.7468,651,0.3974
1,All 21 (unpenalized),0.30,False,640,0,217,326,0.8166,1.0000,0.7468,651,0.3974
2,All 21 (unpenalized),0.50,False,639,1,212,331,0.8199,0.9984,0.7509,637,0.3974
3,All 21 (unpenalized),0.70,False,444,196,117,426,0.7354,0.6938,0.7914,547,0.3974
4,All 21 (unpenalized),0.77,True,286,354,68,475,0.6433,0.4469,0.8079,558,0.3974
5,All 21 (unpenalized),0.90,False,13,627,1,542,0.4691,0.0203,0.9286,630,0.3974
6,Ridge,0.10,False,640,0,217,326,0.8166,1.0000,0.7468,651,0.3980
7,Ridge,0.30,False,640,0,217,326,0.8166,1.0000,0.7468,651,0.3980
8,Ridge,0.50,False,639,1,212,331,0.8199,0.9984,0.7509,637,0.3980
9,Ridge,0.70,False,442,198,117,426,0.7337,0.6906,0.7907,549,0.3980
